# 06 — Matched-marker probe (Table 1)

**CPU. Part A: hours (N_BOOT = 2000; resumable). Part B: minutes.**

Table 1 is produced by **both parts, in order**:

| Column | Source |
|---|---|
| Matched, Null, all three CIs, peak layer | Part A → `results_v3/task1_residual_cis_nboot2000_v3.json` |
| Residual (point estimate) | Part B → `results_v3/task1_residual_pointfix.json`, field `residual_paired` |

Three contrasts per model: **matched** (revision vs filler windows with the same marker word), **null** (filler vs filler, random split; should sit at chance), and the **residual** (matched − null, paired within each draw).

**Prerequisites:** `activations/` (notebook 03) and `activations_filler/` (notebook 05).

In [ ]:
# ---- Setup: Google Drive and the data folder ---------------------------------
import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/gdrive")

# Folder on Drive that holds traces/, labels/, activations/, rejects/ and
# hand_labeling_export/. Change it here if yours lives elsewhere.
DATA_ROOT = Path(os.environ.get("REVISION_DATA_ROOT",
                                "/content/gdrive/MyDrive/llm_planning_study"))
print("DATA_ROOT =", DATA_ROOT)

## Part A — matched, null, bootstrap CIs

In [ ]:
# =============================================================================
# 06 PART A -- MATCHED-MARKER PROBE (Table 1: matched, null, all CIs)
#
# For each model: pick the layer with the highest matched accuracy (revision
# vs filler windows, centre position = token before the marker), then
# bootstrap matched, null (filler vs filler) and their within-replicate
# difference, N_BOOT = 2000.
#
#   * Group-aware CV (StratifiedGroupKFold on original event ids), so a
#     resampled duplicate never sits on both sides of a fold.
#   * StandardScaler -> PCA(50, fit inside each fold) -> logistic regression.
#   * Matched / null point estimates are the mean of N_PT_DRAWS = 30 balanced
#     draws.
#   * The full per-layer accuracy profile is saved with each model.
#
# The residual point estimate written here is the UNPAIRED difference
# (matched_pt - null_pt). Part B recomputes it paired, which is what Table 1
# reports.
#
# CPU, hours. Resumable: models already in OUTFILE are skipped.
# =============================================================================

from pathlib import Path
GDRIVE_BASE = DATA_ROOT
assert (GDRIVE_BASE / "activations").exists(),        "activations/ missing -- bad mount"
assert (GDRIVE_BASE / "activations_filler").exists(), "fillers missing -- bad mount"
print("Drive mounted, data visible.")

import json, gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore")

ACTIV_BASE  = GDRIVE_BASE / "activations"
FILLER_BASE = GDRIVE_BASE / "activations_filler"
RESULTS_DIR = GDRIVE_BASE / "results_v3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

CENTER       = 30
SEED         = 42
N_PCA        = 50          # pre-committed headline dimensionality
MAX_ITER     = 2000
N_BOOT       = 2000
N_PT_DRAWS   = 30          # averaged point-estimate draws
FLAT_WINDOW  = 0.02        # layers within this of peak are "flat with it"

OUTFILE = RESULTS_DIR / "task1_residual_cis_nboot2000_v3.json"

In [ ]:
def load_rev(m):
    rev = []
    for p in sorted((ACTIV_BASE / m).glob("*.npz")):
        rw = np.load(p).get("revision_windows")
        if rw is not None:
            rev.extend(rw[i] for i in range(rw.shape[0]))
    return rev

def load_fil(m):
    fil = []
    for p in sorted((FILLER_BASE / m).glob("*.npz")):
        w = np.load(p).get("filler_windows")
        if w is not None:
            fil.extend(w[i] for i in range(w.shape[0]))
    return fil


# -----------------------------------------------------------------------
# cv_accuracy threads original-event-id groups through every
# subsampling step and uses StratifiedGroupKFold. Every array of features
# passed in (Xp, Xn) must be accompanied by a same-length array of original
# event IDs (gp, gn) so duplicates introduced by bootstrap resampling never
# get split across train/test.
# -----------------------------------------------------------------------
def cv_accuracy(Xp, Xn, gp, gn, seed):
    """Balanced 5-fold, group-aware CV accuracy. PCA fit inside each fold."""
    n = min(len(Xp), len(Xn))
    if n < 20:
        return 0.5
    rng = np.random.RandomState(seed)
    idx_p = rng.choice(len(Xp), n, False)
    idx_n = rng.choice(len(Xn), n, False)
    X = np.vstack([Xp[idx_p], Xn[idx_n]]).astype(np.float32)
    X = np.nan_to_num(X, nan=0., posinf=0., neginf=0.)
    y = np.array([1]*n + [0]*n)
    g = np.concatenate([gp[idx_p], gn[idx_n]])

    ncomp = min(N_PCA, len(y) - 1)
    pipe = Pipeline([
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=ncomp, random_state=seed)),
        ("clf", LogisticRegression(max_iter=MAX_ITER, random_state=seed)),
    ])
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    accs = []
    for tr, te in cv.split(X, y, groups=g):
        pipe.fit(X[tr], y[tr])
        accs.append((pipe.predict(X[te]) == y[te]).mean())
    return float(np.mean(accs))


def averaged_point_estimate(Xp, Xn, gp, gn, base_seed, n_draws=N_PT_DRAWS):
    """Point estimate = mean cv_accuracy over n_draws independent
    balanced subsample draws, so it isn't hostage to one lucky split."""
    vals = [cv_accuracy(Xp, Xn, gp, gn, base_seed + 10_000 + d) for d in range(n_draws)]
    return float(np.mean(vals)), vals


def peak_layer(rev, fil, nl):
    """Returns the peak layer and the full per-layer accuracy profile."""
    profile = []
    best_l, best_a = 0, 0.0
    rev_ids = np.arange(len(rev))
    fil_ids = np.arange(len(fil)) + 10**7   # disjoint id space from revisions
    for l in range(nl):
        Xp = np.array([w[l, CENTER] for w in rev], np.float32)
        Xn = np.array([w[l, CENTER] for w in fil], np.float32)
        a = cv_accuracy(Xp, Xn, rev_ids, fil_ids, SEED)
        profile.append(a)
        if a > best_a:
            best_a, best_l = a, l
    near_peak = [l for l, acc in enumerate(profile) if best_a - acc < FLAT_WINDOW]
    is_flat = len(near_peak) > 3
    return best_l, best_a, profile, near_peak, is_flat


def bootstrap_all(rev, fil, L):
    """
    Returns point estimates (averaged over draws) and percentile CIs for
    matched, null, and residual. Bootstrap resamples EVENTS with replacement
    and REFITS both probes each iteration; original event IDs are carried
    through the resample step so cv_accuracy's group-aware split
    can prevent the same event landing on both sides of a fold.
    """
    Xr = np.array([w[L, CENTER] for w in rev], dtype=np.float32)
    Xf = np.array([w[L, CENTER] for w in fil], dtype=np.float32)
    gid_r = np.arange(len(Xr))
    gid_f = np.arange(len(Xf)) + 10**7

    # ---- point estimates (averaged over N_PT_DRAWS balanced draws) ----
    matched_pt, matched_draws = averaged_point_estimate(Xr, Xf, gid_r, gid_f, SEED)

    permf = np.random.RandomState(SEED).permutation(len(Xf))
    hf    = len(Xf) // 2
    Xf_a, Xf_b_half = Xf[permf[:hf]], Xf[permf[hf:2*hf]]
    gf_a, gf_b_half = gid_f[permf[:hf]], gid_f[permf[hf:2*hf]]
    # null halves are disjoint filler subsets -> give them disjoint synthetic
    # group ids too, since they're different physical events by construction
    null_pt, null_draws = averaged_point_estimate(Xf_a, Xf_b_half, gf_a, gf_b_half, SEED)

    resid_pt = matched_pt - null_pt

    # ---- bootstrap ----
    rng = np.random.RandomState(SEED)
    matched_b, null_b, resid_b = [], [], []
    for b in range(N_BOOT):
        ir  = rng.choice(len(Xr), len(Xr), replace=True)
        iff = rng.choice(len(Xf), len(Xf), replace=True)
        Xr_b, gr_b = Xr[ir], gid_r[ir]
        Xf_b, gf_b = Xf[iff], gid_f[iff]

        m_b = cv_accuracy(Xr_b, Xf_b, gr_b, gf_b, SEED + b)              # refit, group-aware

        pf  = np.random.RandomState(SEED + b).permutation(len(Xf_b))
        h   = len(Xf_b) // 2
        n_b = cv_accuracy(Xf_b[pf[:h]], Xf_b[pf[h:2*h]],
                           gf_b[pf[:h]], gf_b[pf[h:2*h]], SEED + b)      # refit, group-aware

        matched_b.append(m_b)
        null_b.append(n_b)
        resid_b.append(m_b - n_b)

        if (b + 1) % 250 == 0:
            print(f"      boot {b+1}/{N_BOOT}")

    def ci(arr):
        lo, hi = np.percentile(arr, [2.5, 97.5])
        return float(lo), float(hi)

    m_lo, m_hi = ci(matched_b)
    n_lo, n_hi = ci(null_b)
    r_lo, r_hi = ci(resid_b)

    return {
        "matched":      matched_pt, "matched_ci":  [m_lo, m_hi],
        "null_fil":     null_pt,    "null_ci":     [n_lo, n_hi],
        "residual":     resid_pt,   "residual_ci": [r_lo, r_hi],
        "excludes_zero":        bool(r_lo > 0),
        "null_brackets_chance": bool(n_lo <= 0.50 <= n_hi),
        "intervals_disjoint":   bool(m_lo > n_hi),
        # Acceptance-criterion helper: does the point estimate land
        # comfortably inside its own bootstrap interval, or hug an edge?
        "matched_pt_ok": bool(m_lo < matched_pt < m_hi),
        "null_pt_ok":    bool(n_lo < null_pt < n_hi),
        "n_boot": N_BOOT,
        "n_pt_draws": N_PT_DRAWS,
    }

In [ ]:
# ---- resume support -------------------------------------------------------
results = {}
if OUTFILE.exists():
    results = json.load(open(OUTFILE))
    print(f"Resuming -- already done: {list(results.keys())}")

In [ ]:
for m in MODELS:
    if m in results:
        print(f"  {m}: already done, skipping")
        continue

    print(f"\n{'='*60}\n{m}\n{'='*60}")
    rev = load_rev(m)
    fil = load_fil(m)
    nl  = rev[0].shape[0]
    print(f"  revisions={len(rev)}  fillers={len(fil)}")

    L, a, profile, near_peak, is_flat = peak_layer(rev, fil, nl)
    print(f"  peak layer = {L}/{nl}  (matched acc={a:.3f})")
    print(f"  layers within {FLAT_WINDOW} of peak: {near_peak}")
    print(f"  {'FLAT PROFILE -- argmax is likely not a real peak' if is_flat else 'looks like a genuine peak'}")

    print(f"  bootstrapping (N_BOOT={N_BOOT}, group-aware CV, "
          f"point estimate averaged over {N_PT_DRAWS} draws)...")
    r = bootstrap_all(rev, fil, L)

    print(f"    matched  = {r['matched']:.3f}  "
          f"[{r['matched_ci'][0]:.3f}, {r['matched_ci'][1]:.3f}]  pt_inside_ci={r['matched_pt_ok']}")
    print(f"    null-fil = {r['null_fil']:.3f}  "
          f"[{r['null_ci'][0]:.3f}, {r['null_ci'][1]:.3f}]"
          f"   brackets 0.50: {r['null_brackets_chance']}  pt_inside_ci={r['null_pt_ok']}")
    print(f"    RESIDUAL = {r['residual']:+.3f}  "
          f"[{r['residual_ci'][0]:+.3f}, {r['residual_ci'][1]:+.3f}]"
          f"   excludes 0: {r['excludes_zero']}")
    print(f"    matched/null intervals disjoint: {r['intervals_disjoint']}")

    results[m] = {"peak_layer": L, "layer_profile": profile,
                  "near_peak_layers": near_peak, "flat_profile": is_flat,
                  "n_rev": len(rev), "n_fil": len(fil), **r}
    json.dump(results, open(OUTFILE, "w"), indent=2)
    del rev, fil
    gc.collect()

In [ ]:
# ---- SUMMARY --------------------------------------------------------------
print("\n" + "#"*78)
print(f"#  MATCHED-MARKER PROBE (N_BOOT={N_BOOT}, group-aware CV, averaged point estimate)")
print("#"*78)
print(f"\n  {'Model':<16} {'matched [95% CI]':>24} {'null [95% CI]':>24} "
      f"{'residual [95% CI]':>26}")
print("  " + "-"*92)
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    short = m.replace("DeepSeek-R1-Distill-", "R1-")
    print(f"  {short:<16} "
          f"{r['matched']:.3f} [{r['matched_ci'][0]:.3f},{r['matched_ci'][1]:.3f}]".rjust(24)
          + "  " +
          f"{r['null_fil']:.3f} [{r['null_ci'][0]:.3f},{r['null_ci'][1]:.3f}]".rjust(22)
          + "  " +
          f"{r['residual']:+.3f} [{r['residual_ci'][0]:+.3f},{r['residual_ci'][1]:+.3f}]".rjust(24))

print("\n  ACCEPTANCE CRITERIA")
print("  " + "-"*76)
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    short = m.replace("DeepSeek-R1-Distill-", "R1-")
    print(f"    {short:<16} pt-in-CI(matched/null): "
          f"{str(r['matched_pt_ok']):>5}/{str(r['null_pt_ok']):>5}   "
          f"residual excl. 0: {str(r['excludes_zero']):>5}   "
          f"null brackets .50: {str(r['null_brackets_chance']):>5}   "
          f"flat peak profile: {str(r['flat_profile']):>5}")

n_pass    = sum(results[m]["excludes_zero"] for m in results)
n_null_ok = sum(results[m]["null_brackets_chance"] for m in results)
n_pt_ok   = sum(results[m]["matched_pt_ok"] and results[m]["null_pt_ok"] for m in results)
print(f"\n  {n_pass}/{len(results)} residual CIs exclude zero.")
print(f"  {n_null_ok}/{len(results)} null CIs bracket chance (0.50) as they should.")
print(f"  {n_pt_ok}/{len(results)} models have BOTH point estimates comfortably inside their CIs.")
if n_pass == len(results) and n_null_ok == len(results) and n_pt_ok == len(results):
    print("  >>> All acceptance criteria met (duplicate-leakage-free).")
else:
    print("  >>> One or more acceptance criteria failed -- see per-model detail above "
          "before writing anything up.")

## Part B — paired residual point estimate

In [ ]:
# =============================================================================
# 06 PART B -- PAIRED RESIDUAL POINT ESTIMATE (Table 1: residual column)
#
# Part A's residual point estimate is matched_pt - null_pt, two separately
# averaged marginals, while its CI comes from within-replicate differences.
# They describe different quantities, and for Qwen-7B the point estimate
# fell outside its own CI. This recomputes the point estimate the way the
# bootstrap pairs it: one balanced filler subsample per draw, used for both
# the matched negative class and the null split, differenced within the draw,
# averaged over N_PT_DRAWS = 100 draws, at Part A's saved peak layer.
#
# matched, null and all three CIs are carried over from Part A unchanged.
# Writes results_v3/task1_residual_pointfix.json (the file Table 1 cites:
# column `residual_paired`).
#
# CPU, minutes per model.
# =============================================================================

from pathlib import Path
import json, gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore")

GDRIVE_BASE = DATA_ROOT
ACTIV_BASE  = GDRIVE_BASE / "activations"
FILLER_BASE = GDRIVE_BASE / "activations_filler"

# Part A's results file
V3_RESULTS_PATH = GDRIVE_BASE / "results_v3" / "task1_residual_cis_nboot2000_v3.json"
OUT_PATH        = GDRIVE_BASE / "results_v3" / "task1_residual_pointfix.json"

MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

CENTER     = 30
SEED       = 42
N_PCA      = 50     # must match Part A
MAX_ITER   = 2000   # must match Part A
N_PT_DRAWS = 100    # paired draws (Part A used 30 for its unpaired marginals)


def load_rev(m):
    out = []
    for p in sorted((ACTIV_BASE / m).glob("*.npz")):
        with np.load(p) as z:
            rw = z.get("revision_windows")
            if rw is not None:
                out.extend(rw[i] for i in range(rw.shape[0]))
    return out


def load_fil(m):
    out = []
    for p in sorted((FILLER_BASE / m).glob("*.npz")):
        with np.load(p) as z:
            w = z.get("filler_windows")
            if w is not None:
                out.extend(w[i] for i in range(w.shape[0]))
    return out


def cv_accuracy(Xp, Xn, gp, gn, seed):
    """Identical to Part A's: balanced, group-aware, PCA fit inside each fold."""
    n = min(len(Xp), len(Xn))
    if n < 20:
        return 0.5
    rng = np.random.RandomState(seed)
    ip = rng.choice(len(Xp), n, False)
    inn = rng.choice(len(Xn), n, False)
    X = np.vstack([Xp[ip], Xn[inn]]).astype(np.float32)
    X = np.nan_to_num(X, nan=0., posinf=0., neginf=0.)
    y = np.array([1]*n + [0]*n)
    g = np.concatenate([gp[ip], gn[inn]])
    pipe = Pipeline([
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=min(N_PCA, len(y) - 1), random_state=seed)),
        ("clf", LogisticRegression(max_iter=MAX_ITER, random_state=seed)),
    ])
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    accs = []
    for tr, te in cv.split(X, y, groups=g):
        pipe.fit(X[tr], y[tr])
        accs.append((pipe.predict(X[te]) == y[te]).mean())
    return float(np.mean(accs))


def paired_residual(Xr, Xf, gid_r, gid_f, base_seed, n_draws=N_PT_DRAWS):
    """One balanced filler subsample per draw, used for BOTH the matched
    negative class and the null split -- mirroring what each bootstrap
    replicate does, so the point estimate and the CI describe the same
    quantity."""
    vals = []
    for d in range(n_draws):
        seed = base_seed + 10_000 + d
        rng = np.random.RandomState(seed)
        n = min(len(Xr), len(Xf))
        ir = rng.choice(len(Xr), n, False)
        iff = rng.choice(len(Xf), n, False)
        Xr_d, gr_d = Xr[ir], gid_r[ir]
        Xf_d, gf_d = Xf[iff], gid_f[iff]

        m = cv_accuracy(Xr_d, Xf_d, gr_d, gf_d, seed)

        pf = np.random.RandomState(seed).permutation(len(Xf_d))
        h = len(Xf_d) // 2
        n_ = cv_accuracy(Xf_d[pf[:h]], Xf_d[pf[h:2*h]],
                          gf_d[pf[:h]], gf_d[pf[h:2*h]], seed)
        vals.append(m - n_)
    return float(np.mean(vals)), vals


def main():
    assert V3_RESULTS_PATH.exists(), (
        f"{V3_RESULTS_PATH} not found -- point this at your v3 results JSON.")
    with open(V3_RESULTS_PATH) as f:
        v3 = json.load(f)
    print(f"Loaded v3 results for: {list(v3.keys())}\n")

    out = {}
    for m in MODELS:
        if m not in v3:
            print(f"{m}: not in v3 results, skipping")
            continue
        r = v3[m]
        layer = r["peak_layer"]
        old_resid = r["residual"]
        lo, hi = r["residual_ci"]

        print(f"{'='*66}\n{m}\n{'='*66}")
        print(f"  peak layer {layer} (reused from v3 -- no layer sweep)")
        print(f"  v3 residual {old_resid:+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  "
              f"inside CI: {lo < old_resid < hi}")

        rev, fil = load_rev(m), load_fil(m)
        Xr = np.array([w[layer, CENTER] for w in rev], dtype=np.float32)
        Xf = np.array([w[layer, CENTER] for w in fil], dtype=np.float32)
        gid_r = np.arange(len(Xr))
        gid_f = np.arange(len(Xf)) + 10**7
        print(f"  n_rev={len(Xr)}  n_fil={len(Xf)}  -- computing {N_PT_DRAWS} paired draws...")

        new_resid, draws = paired_residual(Xr, Xf, gid_r, gid_f, SEED)
        inside = lo < new_resid < hi
        print(f"  NEW residual {new_resid:+.4f}  (draw sd {np.std(draws):.4f})")
        print(f"  inside its own CI: {inside}   change {new_resid - old_resid:+.4f}")

        out[m] = {
            "peak_layer": layer,
            "residual_v3_unpaired": old_resid,
            "residual_paired": new_resid,
            "residual_ci": [lo, hi],
            "inside_ci_before": bool(lo < old_resid < hi),
            "inside_ci_after": bool(inside),
            "n_pt_draws": N_PT_DRAWS,
            "draw_sd": float(np.std(draws)),
            # unchanged, carried through for a complete table
            "matched": r["matched"], "matched_ci": r["matched_ci"],
            "null_fil": r["null_fil"], "null_ci": r["null_ci"],
        }
        del rev, fil, Xr, Xf
        gc.collect()

    print(f"\n{'#'*66}\n# CORRECTED RESIDUAL TABLE\n{'#'*66}")
    print(f"{'Model':<32} {'old':>8} {'new':>8}  {'CI':>22}  in CI")
    all_ok = True
    for m, r in out.items():
        lo, hi = r["residual_ci"]
        ok = r["inside_ci_after"]
        all_ok &= ok
        print(f"{m:<32} {r['residual_v3_unpaired']:+.4f} {r['residual_paired']:+.4f}  "
              f"[{lo:+.4f}, {hi:+.4f}]  {ok}")
    print(f"\nAll point estimates inside their own CIs: {all_ok}")
    if not all_ok:
        print("If any remain outside, that is genuine skew rather than the pairing")
        print("mismatch -- worth understanding before publishing, not patching over.")

    with open(OUT_PATH, "w") as f:
        json.dump(out, f, indent=2)
    print(f"\nWrote {OUT_PATH}")
    print("\nNOTE: matched, null, and ALL THREE CIs are unchanged from v3. Only the")
    print("residual POINT ESTIMATE is recomputed. The CIs were already paired")
    print("(bootstrap replicates difference within-replicate), so they were always")
    print("correct -- it was the point estimate that was inconsistent with them.")


if __name__ == "__main__":
    main()